In [1]:
# Step 1: Configure the 10 crosswalk digital objects used in Figure 1 / Table 2.
from __future__ import annotations

from datetime import datetime
from io import StringIO
from pathlib import Path
from functools import lru_cache

import pandas as pd
import requests
from IPython.display import display


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "notebook").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

LOCAL_CROSSWALKS = {
    "azimuth": "https://cdn.humanatlas.io/digital-objects/ctann/azimuth/v1.4/assets/azimuth-crosswalk.csv",
    "celltypist": "https://cdn.humanatlas.io/digital-objects/ctann/celltypist/v1.3/assets/celltypist-crosswalk.csv",
    "deepcelltypes": "https://cdn.humanatlas.io/digital-objects/ctann/deepcelltypes/v1.2/assets/deepcelltypes-crosswalk.csv",
    "deepcelltypes_hubmap": "https://cdn.humanatlas.io/digital-objects/ctann/deepcelltypes-hubmap/v1.2/assets/deepcelltypes-hubmap-crosswalk.csv",
    "frmatch": "https://cdn.humanatlas.io/digital-objects/ctann/frmatch/v1.0/assets/frmatch-crosswalk.csv",
    "pan_human_azimuth": "https://cdn.humanatlas.io/digital-objects/ctann/pan-human-azimuth/v1.2/assets/pan-human-azimuth-crosswalk.csv",
    "popv": "https://cdn.humanatlas.io/digital-objects/ctann/popv/v1.4/assets/popv-crosswalk.csv",
    "ribca": "https://cdn.humanatlas.io/digital-objects/ctann/ribca/v1.0/assets/ribca-crosswalk.csv",
    "stellar": "https://cdn.humanatlas.io/digital-objects/ctann/stellar/v1.0/assets/stellar-crosswalk.csv",
    "vccf": "https://cdn.humanatlas.io/digital-objects/ctann/vccf/v1.2/assets/vccf-crosswalk.csv",
}
SOURCE_LABELS = {
    "azimuth": "Azimuth",
    "celltypist": "CellTypist",
    "deepcelltypes": "DeepCell Types",
    "deepcelltypes_hubmap": "DeepCell Types-HuBMAP",
    "frmatch": "FR-Match",
    "pan_human_azimuth": "Pan-human Azimuth",
    "popv": "popV",
    "ribca": "RIBCA",
    "stellar": "STELLAR",
    "vccf": "CDE Spatial Omics",
}


@lru_cache(maxsize=None)
def get_organ_label(organ_id: str) -> str | None:
    iri = f"http://purl.obolibrary.org/obo/{organ_id.replace(':', '_')}"
    response = requests.get(
        "https://www.ebi.ac.uk/ols4/api/ontologies/uberon/terms",
        params={"iri": iri},
        timeout=30,
    )
    if response.status_code == 200:
        terms = response.json().get("_embedded", {}).get("terms", [])
        if terms:
            return terms[0].get("label")
    return None


In [2]:
# Step 2: Read crosswalks and apply the same organ resolution and Unspecified filter as Figure 1.
def load_crosswalk(url: str, source: str) -> pd.DataFrame:
    response = requests.get(url, timeout=90)
    response.raise_for_status()
    table = pd.read_csv(StringIO(response.text), skiprows=10)
    table["source"] = source
    return table


df_combined = pd.concat(
    [load_crosswalk(url, source) for source, url in LOCAL_CROSSWALKS.items()],
    ignore_index=True,
)
for column in ["Organ_ID", "Annotation_Label_ID", "CL_ID", "CL_Label", "CL_Match"]:
    df_combined[column] = df_combined[column].fillna("").astype(str).str.strip()

organ_lookup = {
    organ_id: get_organ_label(organ_id)
    for organ_id in df_combined["Organ_ID"].loc[df_combined["Organ_ID"].ne("")].unique()
}
df_combined["Organ"] = df_combined["Organ_ID"].map(organ_lookup)
df_combined["Organ"] = df_combined["Organ"].fillna("Unspecified").astype(str)
df_combined = df_combined.loc[df_combined["Organ"].ne("Unspecified")].copy()
df_combined["Tool"] = df_combined["source"].map(SOURCE_LABELS)

print(f"Rows retained after omitting Unspecified organs: {len(df_combined):,}")
display(df_combined.head())


Rows retained after omitting Unspecified organs: 3,457


,Organ_Level,Organ_ID,Annotation_Label,Annotation_Label_ID,CL_Label,CL_ID,CL_Match,source,Unnamed: 7,Organ,Tool
0,Heart_L2,UBERON:0000948,Adipocyte,AZ:0000001,adipocyte,CL:0000136,skos:exactMatch,azimuth,NaN,heart,Azimuth
1,Heart_L2,UBERON:0000948,Arterial Endothelial,AZ:0000002,endothelial cell of artery,CL:1000413,skos:exactMatch,azimuth,NaN,heart,Azimuth
2,Heart_L2,UBERON:0000948,Atrial Cardiomyocyte,AZ:0000003,regular atrial cardiac myocyte,CL:0002129,skos:exactMatch,azimuth,NaN,heart,Azimuth
3,Heart_L2,UBERON:0000948,B,AZ:0000004,B cell,CL:0000236,skos:exactMatch,azimuth,NaN,heart,Azimuth
4,Heart_L2,UBERON:0000948,Capillary Endothelial,AZ:0000005,capillary endothelial cell,CL:0002144,skos:exactMatch,azimuth,NaN,heart,Azimuth


In [3]:
# Step 3: Create Table 2 with unique counts and SKOS relationship percentages.
mapping_keys = ["source", "Organ_ID", "Annotation_Label_ID", "CL_ID", "CL_Match"]
df_mappings = df_combined.drop_duplicates(mapping_keys).copy()
match_types = ("skos:exactMatch", "skos:narrowMatch", "skos:broadMatch")

records = []
for source, group in df_combined.groupby("source", sort=False):
    mappings = df_mappings.loc[df_mappings["source"].eq(source)]
    match_counts = mappings["CL_Match"].value_counts()
    exact = int(match_counts.get("skos:exactMatch", 0))
    narrow = int(match_counts.get("skos:narrowMatch", 0))
    broad = int(match_counts.get("skos:broadMatch", 0))
    total_skos = exact + narrow + broad
    if total_skos != len(mappings):
        raise ValueError(f"Unexpected non-SKOS mapping type for {source}")

    records.append({
        "Tool": SOURCE_LABELS[source],
        "Number of Unique Organ": group.loc[group["Organ_ID"].ne(""), "Organ_ID"].nunique(),
        "Number of Unique CL ID": group.loc[group["CL_ID"].ne(""), "CL_ID"].nunique(),
        "Number of Unique CL Label": group.loc[group["CL_Label"].ne(""), "CL_Label"].nunique(),
        "Percent Exact Match": round(exact / total_skos * 100, 2),
        "Percent Narrow Match": round(narrow / total_skos * 100, 2),
        "Percent Broad Match": round(broad / total_skos * 100, 2),
    })

table_2 = pd.DataFrame(records).sort_values("Tool", key=lambda values: values.str.casefold(), kind="stable")
print(
    "Percentages use unique Organ_ID + Annotation_Label_ID + CL_ID + CL_Match relationships "
    "after omitting Unspecified organs."
)
display(table_2)

output_dir = PROJECT_ROOT / "outputs" / "table-2-tool-summary-skos-percentages" / datetime.now().strftime("%Y%m%dT%H%M%S")
output_dir.mkdir(parents=True, exist_ok=False)
output_path = output_dir / "table-2-tool-summary-skos-percentages.csv"
table_2.to_csv(output_path, index=False, encoding="utf-8")


Percentages use unique Organ_ID + Annotation_Label_ID + CL_ID + CL_Match relationships after omitting Unspecified organs.


,Tool,Number of Unique Organ,Number of Unique CL ID,Number of Unique CL Label,Percent Exact Match,Percent Narrow Match,Percent Broad Match
0,Azimuth,20,301,482,57.79,42.21,0.00
9,CDE Spatial Omics,11,126,189,76.80,20.20,3.00
1,CellTypist,16,268,443,57.56,42.44,0.00
2,DeepCell Types,1,29,29,100.00,0.00,0.00
3,DeepCell Types-HuBMAP,1,40,40,97.56,2.44,0.00
4,FR-Match,1,53,61,59.42,37.68,2.90
5,Pan-human Azimuth,1,241,440,49.33,50.45,0.22
6,popV,24,157,157,99.78,0.22,0.00
7,RIBCA,1,15,15,100.00,0.00,0.00
8,STELLAR,1,21,25,84.00,16.00,0.00
